# STAT 764 · Meeting 2 — The smallest honest pipeline

**Thursday, September 10 · Concept 10 min · Studio 45 min**

On Tuesday you reported a number you could not have known was right. Most of
the room was optimistic by something between 0.10 and 0.45.

Nobody was careless. The number you reported answered a real question — *how
well does this model describe the 120 houses I fit it to?* — it just wasn't the
question anyone actually cares about.

Today you build the smallest machine that answers the right one.

## Finding the course files

Every notebook from here on starts with these four lines. They walk up from
wherever you opened the notebook until they find the repo, so the same code
works from `course/` and from your copy in `work/`.

The helpers live in `course/stat764.py` — a plain Python module, not a
notebook. **Anything reused across meetings goes in a module.** You will hear
this again; it is the reason your capstone still runs in December.

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
root = next(p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists())
sys.path.insert(0, str(root / "course"))

from stat764 import load

In [ ]:
import numpy as np
import pandas as pd

ames = load("ames.csv")
print(f"{ames.shape[0]} sales, {ames.shape[1]} columns — the full De Cock data")
print(f"Tuesday you had {120} rows and 14 columns.")

## 1. Every model is the same three verbs

scikit-learn has one interface. Learn it once and every method in this course —
ridge, trees, forests, boosting, the foundation model in November — is the same
three lines with a different name at the top.

| verb | what it does |
|---|---|
| `.fit(X, y)` | learn from predictors `X` and outcomes `y` |
| `.predict(X)` | produce a prediction per row |
| `.predict_proba(X)` | for classifiers: a probability per class |

`X` is a table, `y` is a column. That is the whole API.

In [ ]:
from sklearn.linear_model import LinearRegression

simple = ames[["Gr_Liv_Area", "Overall_Qual", "Year_Built"]]
price = ames["SalePrice"]

model = LinearRegression()
model.fit(simple, price)

print("coefficients:", dict(zip(simple.columns, model.coef_.round(1))))
print("first 3 predictions:", model.predict(simple.head(3)).round(0))

## 2. The honest number needs data the model never saw

`train_test_split` holds part of the data back. You fit on one part and score
on the other — which is exactly what Tuesday's holdout file did to you, except
now you do it to yourself, on purpose, before anyone else can.

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    simple, price, test_size=0.25, random_state=764
)

model = LinearRegression().fit(X_train, y_train)

print(f"  scored on data it was fit to:  {r2_score(y_train, model.predict(X_train)):.3f}")
print(f"  scored on data it never saw:   {r2_score(y_test, model.predict(X_test)):.3f}")

With three well-behaved predictors the gap is small. That is not reassurance —
Tuesday's gap was 0.19 with fourteen predictors and 120 rows, and a full tree
would have shown you 1.000 against 0.55. **The gap grows with model
flexibility and shrinks with sample size.** You cannot know which regime you
are in by looking at the training number, which is the entire problem.

## 3. Where it goes wrong: preprocessing outside the split

Real data needs work before modeling — scaling, dummy coding, filling in
missing values. The obvious order is to clean everything, then split.

The obvious order is wrong. Watch what `StandardScaler` uses:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(simple)
print("The mean it subtracts is computed from ALL", len(simple), "rows:")
print(pd.Series(scaler.mean_, index=simple.columns).round(1))

If you scale before splitting, the mean of your test set is baked into the
numbers your model trains on. Your test set is no longer data the model never
saw — it leaked in through the preprocessing.

Here it is a small effect. Impute a missing value with a column mean computed
over everything, or select features by correlation with the outcome over
everything, and it stops being small. **Meeting 4 is an hour of this.**

The fix is not "remember to be careful." The fix is structural.

## 4. `Pipeline`: preprocessing becomes part of the model

A `Pipeline` chains steps into one object with the same three verbs. When you
call `.fit()`, every step learns from the training data only. When you call
`.predict()`, every step applies what it learned.

**This is the single most important object in the course.** Once preprocessing
lives inside the pipeline, the leakage in §3 is not something you have to
remember to avoid — it becomes something you would have to work to cause.

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression()),
])

pipe.fit(X_train, y_train)
print(f"  R-squared on unseen data: {r2_score(y_test, pipe.predict(X_test)):.3f}")
print(f"\n  the scaler inside was fit on the {len(X_train)} training rows only,")
print(f"  never on the {len(X_test)} test rows — because .fit() only ever saw X_train")

## 5. `ColumnTransformer`: numbers and categories need different treatment

Ames has both. `Neighborhood` is not a number, and `Gr_Liv_Area` should not be
one-hot encoded. `ColumnTransformer` routes each group of columns to its own
preprocessing, and the whole assembly still behaves like one model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric = ["Gr_Liv_Area", "Lot_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF"]
categorical = ["Neighborhood", "Central_Air"]

prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

full = Pipeline([("prep", prep), ("model", LinearRegression())])

X = ames[numeric + categorical]
y = ames["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=764)

full.fit(X_train, y_train)
print(f"  R-squared on unseen data: {r2_score(y_test, full.predict(X_test)):.3f}")

Note `handle_unknown="ignore"`. A neighborhood that appears only in the test
set would otherwise crash `.predict()`. Every real deployment meets a category
it was not trained on.

Note also `SimpleImputer` **inside** the pipeline. The median it fills with is
computed from training rows only. Do that outside and you have leaked.

### What one-hot encoding actually does

It is worth seeing, because you will use it all semester. `Neighborhood` is a word,
and the model does arithmetic — so the encoder makes **one column per level**, holding
a 1 where that level occurs and 0 everywhere else.

In [ ]:
# four rows, deliberately from four different neighborhoods
sample = ames[["Neighborhood"]].drop_duplicates().head(4)

peek = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
out = pd.DataFrame(peek.fit_transform(sample),
                   columns=[c.replace("Neighborhood_", "") for c in peek.get_feature_names_out()],
                   index=sample.index).astype(int)

print(sample.to_string(), "\n\n becomes\n")
print(out.to_string())
print("\n one column per level; exactly one 1 in each row")

**You have used this for years.** When you write `lm(price ~ neighborhood)` in R with a
factor, R does exactly this behind the scenes. The difference is only that here you
have to say it out loud.

Two things worth knowing before you use it on real data.

**1. It costs you columns.** Ames has 28 neighborhoods, so those two categorical
columns become 30:

In [ ]:
enc = OneHotEncoder(handle_unknown="ignore").fit(ames[["Neighborhood", "Central_Air"]])
print(f"  2 columns in  ->  {len(enc.get_feature_names_out())} columns out")

That is fine here. On a medical dataset with ICD-9 diagnosis codes it would be
hundreds, and one-hot stops being the obvious choice.

**2. R drops a reference level. `OneHotEncoder` does not.**

R gives you 27 columns for 28 neighborhoods, because with an intercept the 28th is
redundant — its column is one minus the sum of the others. `OneHotEncoder` keeps all
28 by default, so the design matrix is rank deficient.

For *prediction* this does not matter: the fitted values are identical either way, and
`LinearRegression` handles the redundancy without complaint. For *coefficients* it does
matter — they are no longer unique, so do not read them one at a time. Which is a
theme you have already met if you looked at the reference notebook from Meeting 1.

**Other options exist**, and which one is right depends on the column:

| | when |
|---|---|
| **One-hot** | few levels, no natural order. Our default. |
| **Ordinal** | the order is real — `Poor < Fair < Good` — and you want to keep it |
| **Leave it to the model** | some gradient-boosting libraries take raw categories |

`Overall_Qual` is a good test of your judgment: it runs 1–10 and is already a number,
so we never encoded it. Was that right?

---

## Studio

Build the smallest honest pipeline you can defend, end to end, on the full Ames
data. "Smallest" is a real constraint — this is not a competition and you are
not trying to win.

Your pipeline must:

1. split before anything else touches the data,
2. do all preprocessing inside the pipeline,
3. beat the mean baseline,
4. report a number you would be willing to put on the board.

Pick your own columns from the 82 available. `ames.dtypes` will tell you what
is numeric.

In [ ]:
# YOUR CODE HERE
#
# my_numeric = [...]
# my_categorical = [...]
# my_pipe = Pipeline([...])

## One more verb: `predict_proba`

Classifiers add a fourth verb. We are not doing classification properly until
October, but you should see it once now, because a classifier is a *probability
model plus a decision rule* and almost every mistake in October comes from
conflating those two things.

In [ ]:
from sklearn.linear_model import LogisticRegression

expensive = (ames["SalePrice"] > ames["SalePrice"].median()).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, expensive, test_size=0.25, random_state=764
)

clf = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])
clf.fit(Xc_train, yc_train)

probs = clf.predict_proba(Xc_test)[:, 1]
labels = clf.predict(Xc_test)

print(f"  probabilities for the first 5 houses: {probs[:5].round(3)}")
print(f"  the labels those became:              {labels[:5]}")
print(f"\n  .predict() applied a threshold of 0.5 that nobody chose deliberately.")
print(f"  Hold that thought until October 8.")

## Optional, if you finished early: make a model out of nothing

Today's leak was small — with linear regression, scaling barely moves the number.
Fair to wonder whether any of this actually matters.

It does. Here is the same structural mistake, made with a step that *chooses* things.

Below, `y` is **pure noise**. It has nothing to do with any column. An honest analysis
must report an R-squared of about zero, because there is nothing there.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score, KFold

rng = np.random.default_rng(0)
X_noise = pd.DataFrame(rng.normal(size=(120, 400)))
y_noise = pd.Series(rng.normal(size=120))        # independent of every single column

cv = KFold(5, shuffle=True, random_state=0)

# THE MISTAKE: pick the 8 best-correlated columns using ALL the rows, then cross-validate
picked = SelectKBest(f_regression, k=8).fit(X_noise, y_noise)
leaked = cross_val_score(LinearRegression(), picked.transform(X_noise), y_noise, cv=cv)

# HONEST: selection is a step in the pipeline, so it happens inside every fold
honest_pipe = Pipeline([("select", SelectKBest(f_regression, k=8)),
                        ("model", LinearRegression())])
honest = cross_val_score(honest_pipe, X_noise, y_noise, cv=cv)

print(f"  truth                          R-squared  0.000")
print(f"  selected outside the pipeline  R-squared {leaked.mean():+.3f}   <- from nothing")
print(f"  selection inside the pipeline  R-squared {honest.mean():+.3f}")

The first number is not a small distortion. **There is no signal in that data at
all**, and the leaked analysis reports a fifth of the variance explained.

Why it is so much worse than scaling: a scaler *learns two numbers*. A selector
*makes a choice* — and if it chooses while looking at the outcome across all your
rows, it will find the eight columns that happen to correlate with `y` in this
particular sample. Your test set cannot catch it, because the test set is part of
where those correlations were found.

Same structural error as scaling before the split. Very different blast radius.

**Meeting 4 is an hour of this**, on real data, with eight workflows to sort into
honest and not.

---

## Before you close this: Restart & Run All

**Run → Restart Kernel and Run All Cells.** If it does not run top to bottom in
a fresh kernel, it does not work — you just have a result stored in memory that
you cannot reproduce.

This is not fussiness. Out-of-order execution is *the* canonical notebook
failure, and "Restart & Run All succeeds" is the definition of done for every
lab and every capstone stage in this course. It is a grading criterion.

## Compare

Put your R-squared on the board next to your column count.

1. Did more columns mean a better number? Always?
2. Two people used the same columns and got different numbers. Why?
3. Whose pipeline would still run if I handed it a house from a neighborhood
   nobody has seen?

## Exit ticket

> **Name one thing your pipeline does that would be wrong to do outside it,
> and say what it would leak.**

---

## Also in this neighborhood

📗 **Custom transformers.** When the step you want is not in scikit-learn, wrap
it: `FunctionTransformer` for anything stateless, or subclass `TransformerMixin`
when it needs to *learn* something on `fit`. This is how a real project keeps
domain-specific cleaning inside the pipeline instead of in a script above it.

📗 **Keeping your column names.** `sklearn.set_config(transform_output="pandas")`
makes transformers return DataFrames instead of bare arrays. Debugging a
`ColumnTransformer` is dramatically easier when the output still knows what its
columns are called.

🚫 **Formula interfaces** — `patsy`, `formulaic`, or `statsmodels`'
`smf.ols("SalePrice ~ Gr_Liv_Area + Neighborhood", data=ames)`. These will look
very attractive to you, because they are R's interface and they are genuinely
convenient.

We avoid them for prediction work for one specific reason: a formula builds the
design matrix from **the whole data frame you hand it**, which quietly encourages
doing the encoding *before* the split. That is the exact pattern Meeting 4 is
about. Writing `X` and `y` explicitly is clunkier, and it keeps the split
boundary somewhere you can see it.

They remain the right tool when you are doing inference rather than prediction —
which is most of what you have done until now, and is why they feel natural.

🎓 **Serving a model.** Getting a fitted pipeline out of your notebook and behind
an API — versioning, schema validation, what happens when a category shows up
that you have never seen. Adjacent to the `handle_unknown="ignore"` question you
hit today; a course of its own.